# L17 — Token economics, model routing and capacity planning
**Objectives 2, 4, 15.**

**Northfield Grocers context:** finance has asked what the customer assistant will cost per conversation, per month and at the Christmas peak, and whether routing simple questions to a small self-hosted model beats sending everything to a hosted frontier model. The answer must be a formula-driven model the product owner can defend, built from measured numbers.

**Retail use cases:** cost per resolved customer conversation; GPU replica plan for Saturday and holiday peaks; the routing policy between a small self-hosted model and a hosted model.

**Platform:** any (fully executable in SMOKE mode). The rates below are placeholders — replace with measured throughput (L03) and the provider's current price list.

**Done means:** cost per conversation computed for hosted, self-hosted and routed; break-even utilisation found; peak replica count derived from the arrival curve; routing policy justified with numbers.

## Step 1 — Conversation token profile from the request mix

In [ ]:
# === Lab environment header (identical in every lab) ===
import os, sys, json, time, math, shutil, re, subprocess, importlib, importlib.util
import numpy as np, pandas as pd

def ensure_packages(pkgs):
    """Install any missing pip packages into THIS Python (same mechanism as %pip on Databricks) and import them.
    Fresh packages are importable immediately — no restart. Only labs that need extras call this (L02, L08)."""
    missing = [p for p in pkgs if importlib.util.find_spec(p.replace("-", "_")) is None]
    if not missing:
        print("Packages present:", pkgs); return
    print("Installing missing packages into", sys.executable, ":", missing)
    cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        r = subprocess.run(cmd + ["--break-system-packages"], capture_output=True, text=True)   # local system Pythons
    if r.returncode != 0:
        raise ImportError("pip could not install " + str(missing) + ". Ask the admin to add them as cluster libraries "
                          "(Compute → Libraries → PyPI) or use an internal index. pip said: " + r.stderr[-600:])
    importlib.invalidate_caches()
    for p in missing: importlib.import_module(p.replace("-", "_"))
    print("Installed and imported:", missing)

# Mode: "GPU" runs the full lab on Azure GPU compute; "SMOKE" runs the CPU/synthetic path anywhere.
LAB_MODE = os.environ.get("LAB_MODE") or ("GPU" if shutil.which("nvidia-smi") else "SMOKE")

# Data folder: env override → package-relative (../../data) → Unity Catalog volume → search the workspace once
_candidates = [os.environ.get("DATA_DIR"), os.path.abspath(os.path.join(os.getcwd(), "..", "..", "data")), "/Volumes/northfield/llmops/labdata"]
DATA_DIR = next((c for c in _candidates if c and os.path.exists(os.path.join(c, "catalog_items.csv"))), None)
if DATA_DIR is None:
    import glob
    _hits = [h for root in ("/Workspace", "/Volumes", os.path.expanduser("~")) if os.path.isdir(root)
             for h in glob.glob(os.path.join(root, "**", "catalog_items.csv"), recursive=True)][:1]
    DATA_DIR = os.path.dirname(_hits[0]) if _hits else None
if DATA_DIR is None:
    raise FileNotFoundError("Lab data not found. Upload the package's data/ folder to a Unity Catalog volume and set "
                            "os.environ['DATA_DIR'] = '/Volumes/<catalog>/<schema>/<volume>' in a cell above this one.")
def gpu_only(msg):
    """Called wherever a step needs a GPU / model download that the smoke path cannot run."""
    print(f"[{LAB_MODE}] GPU-only step not executed here: {msg}")
def check(cond, msg):
    """Binary 'done means' assertion — prints PASS/FAIL and raises on FAIL so the notebook stops."""
    print(("PASS " if cond else "FAIL ") + msg); assert cond, msg
print(f"LAB_MODE={LAB_MODE}  DATA_DIR={DATA_DIR}  python={sys.version.split()[0]}")

In [ ]:
reqs = pd.DataFrame([json.loads(l) for l in open(os.path.join(DATA_DIR, "request_mix.jsonl"))])
turns_per_conv = 4
profile = dict(prompt_tok=reqs.prompt_tokens.mean() * turns_per_conv, out_tok=reqs.max_new.mean() * 0.6 * turns_per_conv)   # ~60% of max_new used on average (assumption)
print({k: round(v) for k, v in profile.items()})

## Step 2 — Hosted cost per conversation
*Watch:* prices are placeholders; use the provider's list on the day.

In [ ]:
HOSTED = dict(frontier=dict(in_per_M=2.50, out_per_M=10.0), small=dict(in_per_M=0.15, out_per_M=0.60))   # USD per million tokens — PLACEHOLDERS
def hosted_cost(profile, price):
    return profile["prompt_tok"] * price["in_per_M"] / 1e6 + profile["out_tok"] * price["out_per_M"] / 1e6
hc = {k: hosted_cost(profile, v) for k, v in HOSTED.items()}; print({k: round(v, 5) for k, v in hc.items()})
check(hc["frontier"] > hc["small"] * 5, "hosted cost per conversation computed for both tiers")

## Step 3 — Self-hosted cost per conversation and break-even utilisation
*Why:* a GPU costs the same idle or busy. Cost per conversation = GPU $/hr ÷ conversations per hour at the utilisation you actually achieve.

In [ ]:
sys.path.insert(0, os.path.join(DATA_DIR, "..")); from content import GPU_SKUS
gpu_usd_hr = dict((s[0], s[3]) for s in GPU_SKUS)["Standard_NC24ads_A100_v4"]
measured_tok_s = 1800   # PLACEHOLDER: output tokens/s at concurrency 32 from the L03 harness on this SKU
def self_hosted_cost(profile, tok_s, utilisation, usd_hr=gpu_usd_hr):
    conv_per_hr = tok_s * 3600 * utilisation / profile["out_tok"]
    return usd_hr / conv_per_hr
for u in (0.1, 0.3, 0.6, 0.9): print(f"utilisation {u:.0%}: ${self_hosted_cost(profile, measured_tok_s, u):.5f} per conversation")
breakeven = next(u for u in np.arange(0.01, 1.0, 0.01) if self_hosted_cost(profile, measured_tok_s, u) <= hc["small"])
print(f"break-even vs hosted small tier at ~{breakeven:.0%} utilisation")
check(0 < breakeven < 1, "break-even utilisation found")

## Step 4 — Routing policy: small model first, frontier for the hard 20%
*Why:* most shopping questions are simple (stock, aisle, price). Route by a cheap classifier; escalate the rest. Compute blended cost.

In [ ]:
def routed_cost(share_simple, simple_cost, hard_cost): return share_simple * simple_cost + (1 - share_simple) * hard_cost
blend = {s: routed_cost(s, self_hosted_cost(profile, measured_tok_s, 0.6), hc["frontier"]) for s in (0.5, 0.7, 0.8, 0.9)}
print({f"{k:.0%} simple": round(v, 5) for k, v in blend.items()})
check(blend[0.8] < hc["frontier"] * 0.4, "routing 80% of traffic to the small model cuts cost by more than 60%")

## Step 5 — Capacity plan for the peak
*Why:* replicas are sized on the arrival curve, not the average. Saturday 10:00 ≈ 4× Tuesday 15:00; Christmas week ≈ 2× a normal Saturday.

In [ ]:
base_conv_per_hr = 3000
curve = {"Tue 15:00": 1.0, "Sat 10:00": 4.0, "Xmas Sat 10:00": 8.0}
def replicas_needed(conv_per_hr, tok_s, target_util=0.6, min_rep=2):
    need = conv_per_hr * profile["out_tok"] / (tok_s * 3600 * target_util)
    return max(min_rep, math.ceil(need))
plan = {k: replicas_needed(base_conv_per_hr * m, measured_tok_s) for k, m in curve.items()}; print(plan)
monthly = sum(blend[0.8] * base_conv_per_hr * 24 * 30 * 1.6 for _ in [0])   # 1.6 = average load factor over the week (assumption)
print(f"indicative monthly cost at 80% routing: ${monthly:,.0f} (placeholder rates)")
check(plan["Xmas Sat 10:00"] >= plan["Sat 10:00"] >= plan["Tue 15:00"], "replica plan grows with the arrival curve")
print("L17 complete. Replace every PLACEHOLDER with measured numbers before the finance review.")